
## Інструкція з використання:
1. **Керування гармонікою:** Параметри основного сигналу — його розмах, швидкість коливань та зміщення у часі — регулюються повзунками *«Amplitude»*, *«Frequency»* та *«Phase»*.
2. **Регулювання шуму:** Параметри *«Noise Mean»* та *«Noise Covariance»* дають змогу коригувати середнє значення та інтенсивність завад. Оскільки шумова складова фіксована, її малюнок не змінюється хаотично під час налаштування параметрів основної гармоніки.
3. **Робота з фільтром:** Регулятор *«Cutoff Frequency»* визначає інтенсивність обробки сигналу. Чим нижчий цей поріг, тим помітнішим стає згладжування і тим менше високочастотних коливань пропускає фільтр.  
4. **Керування відображенням:** Чекбокс *"Show Noise"* дозволяє вмикати або вимикати показ зашумленого сигналу на графіку
5. **Скидання:** Кнопка *«Reset»* миттєво повертає всі регулятори до їхніх базових значень, що дозволяє швидко скасувати внесені зміни та почати роботу з графіком з нуля.


In [10]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive, FloatSlider, Checkbox, Button, HBox, VBox
from scipy.signal import butter, lfilter
from IPython.display import display

# Параметри часу
t = np.linspace(0, 10, 1000)
fs = 100 

# п.7-8: Фільтр Баттерворта
def butter_lowpass_filter(data, cutoff, fs, order=5):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return lfilter(b, a, data)

# п.5: Фіксований шум
current_noise = np.random.normal(0, np.sqrt(0.1), len(t))

# п.1-4, 9: Логіка малювання
def plot_func(amplitude, frequency, phase, noise_mean, noise_cov, cutoff, show_noise):
    global current_noise
    
    # п.4: Генерація шуму (параметри mean та cov)
    noise = np.random.normal(noise_mean, np.sqrt(noise_cov), len(t))
    
    # п.1: Чиста гармоніка
    pure_signal = amplitude * np.sin(2 * np.pi * frequency * t + phase)
    
    # п.2: Накладання шуму
    noisy_signal = pure_signal + noise if show_noise else pure_signal
    
    # п.8: Фільтрація
    filtered_signal = butter_lowpass_filter(noisy_signal, cutoff, fs)
    
    # Графіки
    plt.figure(figsize=(10, 6))
    plt.plot(t, noisy_signal, color='orange', alpha=0.5, label='Noisy (п.1)')
    plt.plot(t, pure_signal, color='blue', lw=2, label='Pure (п.1)')
    plt.plot(t, filtered_signal, color='red', lw=2, label='Filtered (п.7)')
    
    plt.ylim(-5, 5)
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)
    plt.title("Lab 4: Interactive Signal Filtering")
    plt.show()

# п.9: Слайдери
w_amp = FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description='Amplitude')
w_freq = FloatSlider(value=0.5, min=0.1, max=5.0, step=0.1, description='Frequency')
w_phase = FloatSlider(value=0.0, min=0, max=2*np.pi, step=0.1, description='Phase')
w_mean = FloatSlider(value=0.0, min=-1.0, max=1.0, step=0.1, description='Noise Mean')
w_cov = FloatSlider(value=0.1, min=0.0, max=1.0, step=0.05, description='Noise Cov')
w_cut = FloatSlider(value=2.0, min=0.1, max=10.0, step=0.1, description='Filter Cutoff')
w_show = Checkbox(value=True, description='Show Noise')

# Інтерфейс
interactive_plot = interactive(
    plot_func, 
    amplitude=w_amp, 
    frequency=w_freq, 
    phase=w_phase, 
    noise_mean=w_mean, 
    noise_cov=w_cov, 
    cutoff=w_cut, 
    show_noise=w_show
)

display(interactive_plot)

interactive(children=(FloatSlider(value=1.0, description='Amplitude', max=3.0, min=0.1), FloatSlider(value=0.5…